In [0]:
dbutils.widgets.text("env", "dev")
env = dbutils.widgets.get("env")

In [0]:
qualifying_df=spark.read\
    .option("inferSchema", True)\
        .json("/Volumes/formula1_dev/bronze/demo_volume/source_files/qualifying/")
qualifying_df.display()

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
qualifying_schema = StructType(fields=[StructField("qualifyId", IntegerType(), False),
                                      StructField("raceId", IntegerType(), True),
                                      StructField("driverId", IntegerType(), True),
                                      StructField("constructorId", IntegerType(), True),
                                      StructField("number", IntegerType(), True),
                                      StructField("position", IntegerType(), True),
                                      StructField("q1", StringType(), True),
                                      StructField("q2", StringType(), True),
                                      StructField("q3", StringType(), True),
                                     ])

In [0]:
qualifying_df=spark.read\
    .schema(qualifying_schema)\
        .option("multiLine", True)\
        .json("/Volumes/formula1_dev/bronze/demo_volume/source_files/qualifying/")
qualifying_df.display()

In [0]:
from pyspark.sql.functions import lit, current_timestamp,current_date
qualifying_df1= qualifying_df.withColumnRenamed("driverId", "driver_id")\
    .withColumnRenamed("raceId", "race_id")\
        .withColumnRenamed("constructorId", "constructor_id")\
    .withColumn("ingestion_timestamp", current_timestamp())\
        .withColumn("ingestion_date", current_date())

In [0]:
qualifying_df1.display()

In [0]:
# laptimes_df1.write.mode("overwrite").format("delta").option("")
qualifying_df1.write.mode("overwrite").format("delta")\
    .option("path", "abfss://raw@formula1adls.dfs.core.windows.net/qualifying").saveAsTable(f"formula1_{env}.bronze.qualifying")


In [0]:
%sql
select * from formula1_dev.bronze.qualifying
